In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, brier_score_loss
from sklearn.feature_selection import SelectFromModel, SelectKBest, f_classif, mutual_info_classif, VarianceThreshold, RFECV
from PersonaClassifier import My_training, Dataset, generate_cal_result
from utils.Visualization import generate_cm, generate_auroc, display_auroc
from utils.Models import MyEstimator, MLP, BiLSTMClassifier, IdentityEstimator
from utils.DataProcessor import FeatureSelection
from sklearn.metrics import make_scorer, accuracy_score
print(torch.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
demo= 1000
my_train = My_training(model_list=['mlp','bilstm'], emb_model='roberta-base')
train_set = Dataset('./processed_data/LIWC_pandora_to_big5_v4.csv', my_train.emb_model, my_train.traits, demo) 
selected_features = {}

In [ ]:
for target_col in my_train.traits:
    print(f'{10*"-"} {target_col} {10*"-"}')
    X, y= my_train.prepare_dataset(train_set.X, train_set.contextual_emb, train_set.Y[[target_col]])
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, stratify=y, shuffle=True, random_state=42)
    param_grid = {
        # 'hidden_dim': [128, 256],
        'lr': [0.001, 0.0001, 0.00001],
        # 'epochs': [10, 16],
        # 'batch_size': [8, 16],
        # 'dropout_rate' :[0.3, 0.5]
    }
    scoring = ['accuracy']

    hyperparameters = {
        'input_dim' : X.shape[1],
        'hidden_dim' : 256,
        'output_dim' :1,
        'dropout_rate': 0.3,
        'batch_size': 16,
        'epochs': 32,
        'learning_rate': 0.0001
    }
    
    bilstm = MyEstimator(model_name="bilstm", input_dim=hyperparameters['input_dim'], hidden_dim=hyperparameters['hidden_dim'], dropout_rate=hyperparameters['dropout_rate'], batch_size=hyperparameters['batch_size'], epochs = hyperparameters['epochs'], lr=hyperparameters['learning_rate'], kFold=False)
    # for param, value in bilstm.get_params(deep=True).items():
    #     print(f"{param} -> {value}")

    # # Perform Grid Search
    # # RandomizedSearchCV
    grid_search = GridSearchCV(estimator=bilstm, param_grid=param_grid, cv=3, refit='accuracy', scoring='accuracy', verbose=1)
    grid_search.fit(X_train, y_train)

    # Best parameters and model
    print("Best Parameters:", grid_search.best_params_)
    # print("Best Model:", grid_search.best_estimator_)

    # Evaluate on test data
    best_model = grid_search.best_estimator_
    print("Test Accuracy:", best_model.score(X_test, y_test))

In [ ]:
for target_col in ['cOPN']:
    print(f'{10*"-"} {target_col} {10*"-"}')
    X, Z, y = train_set.X, train_set.contextual_emb, train_set.Y[[target_col]]
    # X, y = my_train.prepare_dataset(train_set.X, train_set.contextual_emb, train_set.Y[[target_col]])
    features = FeatureSelection.get_optimal_features(X, y)
features

In [ ]:
features